In [ ]:
import pandas as pd
import numpy as np
# import os
# os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=true --xla_cpu_multi_thread_eigen_num_threads=4"
import numpyro
numpyro.set_host_device_count(4)
import matplotlib.pyplot as plt

df = pd.read_csv('../source/df.csv')
df

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
def make_fourier_series(t, period, order):
    """
    Args
    ----
    t: array-like, time points at which to evaluate the Fourier series
    period: int, the period of the seasonality
    order: int, the number of Fourier terms to include
    """
    sin_terms = np.array([np.sin(2 * np.pi * i * t / period) for i in range(1, order + 1)])
    cos_terms = np.array([np.cos(2 * np.pi * i * t / period) for i in range(1, order + 1)])
    return np.concatenate((sin_terms, cos_terms), axis=0).transpose(1, 0)

In [ ]:
response = df["sales"].values
# response_norm = response.mean()
response_norm = 1.
y = np.log(response / response_norm)

In [ ]:
x_glb_trend = np.arange(len(y)) / 365.25
x_annual_seas = make_fourier_series(np.arange(len(y)), period=365.25, order=3)
x_weekly_seas = make_fourier_series(np.arange(len(y)), period=7, order=2)
x_seas = np.concatenate((x_annual_seas, x_weekly_seas), axis=1)
print(x_seas.shape)

In [ ]:
lev_sm = 0.001
slp_sm = 0.001
theta = 0.8

In [ ]:
from wunku.models.dlt import (
    run_dlt_model, 
    generate_in_sample_forecast, 
)
from wunku.hyper_tune import (
    generate_forecast_span_samples,
    compute_log_likelihood,
    compute_wbic,
    run_dlt_model_and_compute_wbic,
    hyper_tuning_dlt_with_wbic,
)
from wunku.utils import summarize_posteriors

In [ ]:
posteriors = run_dlt_model(
    y=y,
    x_glb_trend=x_glb_trend,
    x_seas=x_seas,
    lev_sm=lev_sm,
    slp_sm=slp_sm,
    theta=theta,
)

In [ ]:
summary_df = summarize_posteriors(posteriors)
summary_df

In [ ]:
yhat_lower, yhat_mid, yhat_upper = generate_in_sample_forecast(
    posteriors=posteriors,
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(len(y)), response, label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(len(y)), yhat_mid, label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(len(y)), yhat_lower, yhat_upper, alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('Damped Local Trend Forecast')

In [ ]:
from wunku.utils import flatten_front_dim
h=28
yhat_span = generate_forecast_span_samples(posteriors=posteriors, h=h)
loglk = compute_log_likelihood(
    yhat_span=yhat_span,
    sigma=flatten_front_dim(posteriors["sigma"].to_numpy(), n=2),
    y=y,
)
print(f"loglk shape:", loglk.shape)
wbic = compute_wbic(loglk)
print(f"WBIC: {wbic:.3f}")

Now, let's do a optimization run to find the best hyperparameters for the DLT model.

In [ ]:
data = {
    "y": y,
    "x_glb_trend": x_glb_trend,
    "x_seas": x_seas,
}
# test run objective function
run_dlt_model_and_compute_wbic(
    params=(lev_sm, slp_sm, theta),
    data=data,
    h=12,
)

In [ ]:
import logging
logger = logging.getLogger("wunku")
logger.setLevel(logging.INFO)

In [ ]:
opt_result = hyper_tuning_dlt_with_wbic(
    data=data,
    h=84,
    n_calls=20,
    random_state=42,
)

In [ ]:
# Print best result
best_lev_sm = opt_result.x[0]
best_slp_sm = opt_result.x[1]
best_theta = opt_result.x[2]

print(f"Best parameters: lev_sm={best_lev_sm:.4f}, slp_sm={best_slp_sm:.4f}, theta={best_theta:.4f}")
print(f"Best WBIC: {opt_result.fun}")

In [ ]:
from skopt.plots import plot_convergence
plot_convergence(opt_result)
plt.show()

In [ ]:
from skopt.plots import plot_evaluations
plot_evaluations(opt_result)
plt.show()

In [ ]:
from skopt.plots import plot_objective
plot_objective(opt_result)
plt.show()

In [ ]:
posteriors = run_dlt_model(
    y=y,
    x_glb_trend=x_glb_trend,
    x_seas=x_seas,
    lev_sm=best_lev_sm,
    slp_sm=best_slp_sm,
    theta=best_theta,
)

In [ ]:
yhat_lower, yhat_mid, yhat_upper = generate_in_sample_forecast(
    posteriors=posteriors,
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(len(y)), response, label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(len(y)), yhat_mid, label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(len(y)), yhat_lower, yhat_upper, alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('Damped Local Trend Forecast')